In [4]:
"""
CGNT Stage 1 — Temporal Transformer
=====================================
Trains a Transformer+LSTM forecaster on benign hourly sensor data.
Given the past 48 hours of all 54 sensors, predict hour t+1.

This is identical in spirit to XTM's FPDM module, allowing direct
RMSE comparison. The key difference is that our trained transformer
is then FROZEN and used as a residual generator for the GCN in Stage 2.

Architecture (matches XTM Table 3):
  Input        : (48, 54)
  Transformer  : Multi-head attention (nhead=6, d_model=54) x 2 layers
  LSTM x 2     : hidden=128
  FCL          : 128 -> 54 (one output per sensor)

Training data : hourly_dataset.csv  (benign only, 8760 hours)
Split         : 80% train / 20% val  — time-ordered, no shuffle

After training:
  - Saves model weights     : transformer_best.pt
  - Saves normalization     : transformer_norm.npz
  - Saves val predictions   : transformer_val_preds.npz  (for Stage 2)
  - Reports RMSE/MAE/MSE vs XTM baseline

Author: RAAGAVAN CGNT Research
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
import os
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
HOURLY_PATH = "hourly_dataset.xlsx"
WINDOW      = 48        # past hours used as context
BATCH_SIZE  = 64
EPOCHS      = 150
LR          = 1e-3
WARMUP_EPOCHS = 5
D_MODEL     = 54        # matches number of sensors exactly
NHEAD       = 6         # must divide D_MODEL; 54/6=9 ✓
NUM_LAYERS  = 2         # transformer encoder layers
LSTM_HIDDEN = 128
DROPOUT     = 0.1
EARLY_STOP  = 20
RANDOM_SEED = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Device : {DEVICE}")
print(f"Window : {WINDOW}h  |  D_model : {D_MODEL}  |  Heads : {NHEAD}")


# ─────────────────────────────────────────────
# 1.  DATA LOADING & NORMALIZATION
# ─────────────────────────────────────────────

def load_and_normalize(path):
    df = pd.read_excel(path)
    cols = [f'line_{i}' for i in range(1, 41)] + \
           [f'Cons_{i}'  for i in range(1, 15)]
    data = df[cols].values.astype(np.float32)   # (8760, 54)

    # Time-ordered split — NO shuffle to prevent leakage
    n_windows = len(data) - WINDOW
    split     = int(n_windows * 0.8)

    train_raw = data[:split + WINDOW]           # hours 0 … split+48
    val_raw   = data[split:]                    # hours split … end

    # Fit normalization on TRAIN only
    mu  = train_raw.mean(axis=0, keepdims=True)    # (1, 54)
    sig = train_raw.std(axis=0,  keepdims=True) + 1e-8

    train_norm = (train_raw - mu) / sig
    val_norm   = (val_raw   - mu) / sig

    print(f"\nData split:")
    print(f"  Train hours : 0 – {split + WINDOW - 1}  "
          f"({split} windows)")
    print(f"  Val hours   : {split} – {len(data)-1}  "
          f"({len(data) - split - WINDOW} windows)")
    print(f"  mu  range   : [{mu.min():.3f}, {mu.max():.3f}]")
    print(f"  sig range   : [{sig.min():.5f}, {sig.max():.3f}]")

    np.savez("transformer_norm.npz", mu=mu, sig=sig)
    return train_norm, val_norm, mu, sig, split


def make_windows(data_norm, window=WINDOW):
    """
    Slide a window over normalized data.
    X[i] = data[i : i+window]       shape (window, 54)
    Y[i] = data[i+window]           shape (54,)
    """
    X, Y = [], []
    for i in range(len(data_norm) - window):
        X.append(data_norm[i : i + window])
        Y.append(data_norm[i + window])
    return (torch.tensor(np.array(X), dtype=torch.float32),
            torch.tensor(np.array(Y), dtype=torch.float32))


# ─────────────────────────────────────────────
# 2.  MODEL
# ─────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for sequence length T."""

    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x : (B, T, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


class TemporalTransformer(nn.Module):
    """
    Transformer Encoder + 2×LSTM  →  per-sensor prediction.

    Input  : (B, 48, 54)   — normalized sensor readings
    Output : (B, 54)       — predicted next-hour readings (normalized)

    Architecture mirrors XTM Table 3 closely so RMSE is directly
    comparable.
    """

    def __init__(self, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, lstm_hidden=LSTM_HIDDEN,
                 dropout=DROPOUT):
        super().__init__()

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=num_layers)

        # Two LSTM layers
        self.lstm1 = nn.LSTM(d_model, lstm_hidden,
                             batch_first=True)
        self.lstm2 = nn.LSTM(lstm_hidden, lstm_hidden,
                             batch_first=True)

        # Prediction head
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, d_model)   # one output per sensor
        )

    def forward(self, x):
        # x : (B, 48, 54)
        x = self.pos_enc(x)                    # (B, 48, 54)
        x = self.transformer(x)                # (B, 48, 54)
        x, _ = self.lstm1(x)                   # (B, 48, 128)
        x, _ = self.lstm2(x)                   # (B, 48, 128)
        x = x[:, -1, :]                        # last timestep (B, 128)
        return self.head(x)                    # (B, 54)


# ─────────────────────────────────────────────
# 3.  LEARNING RATE SCHEDULE
# ─────────────────────────────────────────────

def get_lr(epoch, warmup, total, base_lr):
    """Linear warmup → cosine annealing (same schedule as GCN v3)."""
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


# ─────────────────────────────────────────────
# 4.  TRAINING LOOP
# ─────────────────────────────────────────────

def train_epoch(model, X, Y, optimizer, criterion, batch_size, device):
    model.train()
    idx   = torch.randperm(len(X))
    total = 0.0
    for start in range(0, len(X), batch_size):
        batch_idx = idx[start : start + batch_size]
        xb = X[batch_idx].to(device)
        yb = Y[batch_idx].to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(batch_idx)
    return total / len(X)


@torch.no_grad()
def evaluate(model, X, Y, criterion, batch_size, device):
    model.eval()
    preds, trues = [], []
    total = 0.0
    for start in range(0, len(X), batch_size):
        xb = X[start : start + batch_size].to(device)
        yb = Y[start : start + batch_size].to(device)
        pred = model(xb)
        total += criterion(pred, yb).item() * len(xb)
        preds.append(pred.cpu())
        trues.append(yb.cpu())
    return (total / len(X),
            torch.cat(preds).numpy(),
            torch.cat(trues).numpy())


# ─────────────────────────────────────────────
# 5.  METRICS  (denormalized, same as XTM)
# ─────────────────────────────────────────────

def report_metrics(preds_norm, trues_norm, mu, sig, split_name):
    """Denormalize and compute RMSE / MSE / MAE — same as XTM Table 5."""
    preds = preds_norm * sig + mu
    trues = trues_norm * sig + mu
    err   = preds - trues
    mse   = (err ** 2).mean()
    rmse  = np.sqrt(mse)
    mae   = np.abs(err).mean()
    print(f"\n{'='*50}")
    print(f"  {split_name} METRICS  (denormalized, MW)")
    print(f"{'='*50}")
    print(f"  RMSE : {rmse:.4f}   (XTM : 1.2635)")
    print(f"  MSE  : {mse:.4f}   (XTM : 2.4992)")
    print(f"  MAE  : {mae:.4f}   (XTM : 0.9866)")
    print(f"{'='*50}")
    return rmse, mse, mae


# ─────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────

def main():
    print("\n" + "="*60)
    print("  CGNT Stage 1 — Temporal Transformer")
    print("  Benign forecasting on IEEE-14 hourly data")
    print("="*60)

    # Data
    train_norm, val_norm, mu, sig, split = load_and_normalize(HOURLY_PATH)
    X_train, Y_train = make_windows(train_norm)
    X_val,   Y_val   = make_windows(val_norm)
    print(f"\nTrain samples : {len(X_train)}")
    print(f"Val samples   : {len(X_val)}")

    # Model
    model = TemporalTransformer().to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameters : {total_params:,}")

    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=LR, weight_decay=1e-4)
    criterion = nn.MSELoss()

    # Training
    print(f"\nTraining for up to {EPOCHS} epochs...")
    print(f"{'Epoch':>6} {'LR':>10} {'Train MSE':>12} "
          f"{'Val MSE':>10} {'Val RMSE':>10}")
    print("-" * 55)

    best_val_loss    = float('inf')
    best_epoch       = 0
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        lr = get_lr(epoch - 1, WARMUP_EPOCHS, EPOCHS, LR)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        train_loss = train_epoch(model, X_train, Y_train,
                                  optimizer, criterion,
                                  BATCH_SIZE, DEVICE)
        val_loss, val_preds, val_trues = evaluate(
            model, X_val, Y_val, criterion, BATCH_SIZE, DEVICE)

        # Denormalized RMSE for monitoring
        val_rmse = float(np.sqrt(
            ((val_preds * sig + mu) - (val_trues * sig + mu)) ** 2
        ).mean())

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>6} {lr:>10.6f} {train_loss:>12.6f} "
                  f"{val_loss:>10.6f} {val_rmse:>10.4f}")

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_epoch       = epoch
            patience_counter = 0
            torch.save(model.state_dict(), "transformer_best.pt")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP:
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(best: epoch {best_epoch})")
                break

    # Final evaluation
    print(f"\nLoading best model from epoch {best_epoch}...")
    model.load_state_dict(
        torch.load("transformer_best.pt", map_location=DEVICE))

    _, val_preds, val_trues = evaluate(
        model, X_val, Y_val, criterion, BATCH_SIZE, DEVICE)

    rmse, mse, mae = report_metrics(
        val_preds, val_trues, mu, sig, "VALIDATION")

    # Save predictions for Stage 2 residual analysis
    np.savez("transformer_val_preds.npz",
             preds_norm=val_preds,
             trues_norm=val_trues,
             mu=mu, sig=sig)
    print("\nSaved: transformer_val_preds.npz")
    print("Saved: transformer_best.pt")
    print("Saved: transformer_norm.npz")

    # Per-sensor RMSE to understand where errors are largest
    print("\n=== PER-SENSOR RMSE (top 10 hardest sensors) ===")
    preds_mw = val_preds * sig + mu
    trues_mw = val_trues * sig + mu
    per_sensor_rmse = np.sqrt(((preds_mw - trues_mw) ** 2).mean(axis=0))
    top10 = np.argsort(per_sensor_rmse)[::-1][:10]
    for s in top10:
        stype = ('line_fwd' if s < 20 else
                 'line_rev' if s < 40 else 'bus_inj')
        print(f"  sensor {s+1:2d} ({stype}): RMSE = {per_sensor_rmse[s]:.4f} MW")

    # Sensor 47 (bus 7) should have RMSE ~0 since it's always 0
    print(f"\n  sensor 47 (bus_7, always 0): "
          f"RMSE = {per_sensor_rmse[46]:.6f} MW  ✓")

    print(f"\n=== SUMMARY ===")
    print(f"  Stage 1 RMSE : {rmse:.4f}  (XTM: 1.2635)")
    if rmse < 1.2635:
        print(f"  >> BEATS XTM forecasting baseline")
    elif rmse < 2.0:
        print(f"  >> Comparable to XTM — acceptable for residual generation")
    else:
        print(f"  >> Higher than XTM — check training or increase epochs")
    print(f"\n  Stage 2 is ready to proceed.")
    print(f"  Use transformer_best.pt + transformer_norm.npz as frozen backbone.")


if __name__ == '__main__':
    main()

Device : cuda
Window : 48h  |  D_model : 54  |  Heads : 6

  CGNT Stage 1 — Temporal Transformer
  Benign forecasting on IEEE-14 hourly data

Data split:
  Train hours : 0 – 7016  (6969 windows)
  Val hours   : 6969 – 8759  (1743 windows)
  mu  range   : [-177.656, 121.019]
  sig range   : [0.00000, 70.809]

Train samples : 6969
Val samples   : 1743

Model parameters : 321,170

Training for up to 150 epochs...
 Epoch         LR    Train MSE    Val MSE   Val RMSE
-------------------------------------------------------
     1   0.000200     0.676424   0.271254     4.5388
     5   0.001000     0.058339   0.035204     1.3394
    10   0.000998     0.037079   0.020991     1.1261
    15   0.000991     0.028650   0.015468     0.9116
    20   0.000977     0.026563   0.023826     1.3328
    25   0.000958     0.023274   0.013560     0.8712
    30   0.000934     0.021240   0.013378     0.8710
    35   0.000905     0.020621   0.011738     0.7596
    40   0.000870     0.020265   0.011331     0.7864
